# Train BGRL / DGI / MVGRL on **pokec**

clusters=10 (no labels → modularity only). `region=` slices to one region for fast iteration; `None` = full 1.63M users.

Shared models live in `gclib.py`; the loader lives in `train_pokec.py`. This notebook just wires them together so you can iterate on the GPU.

In [ ]:
import gclib, common, importlib
import train_pokec as loader
importlib.reload(gclib); importlib.reload(common); importlib.reload(loader)
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# --- load graph (symmetrized edges + interest-text/profile features) ---
# region='zilinsky kraj' for a small slice; None = full graph (encode on GPU, cached after)
data, labels = loader.load_data(region=None)
print(data)

In [ ]:
# --- config: fixed encoder/dim across all three models for a fair comparison ---
cfg = gclib.Cfg(dim=256, layers=2, epochs=500)
cfg.clusters = 10
cfg

In [ ]:
# MVGRL diffusion view is CPU preprocessing — computed once here, reused below.
# For big graphs cache it: torch.save((data.diff_edge_index, data.diff_edge_weight), 'diff.pt')
results = gclib.run_all(data, cfg, labels=labels, models=['bgrl','dgi','mvgrl'])

In [ ]:
summary = gclib.save(results, './out/pokec', 'pokec')
summary